In [5]:
import pandas as pd

GNAF = "./GNAF/Standard/"   # to find accurate geo path for mapping 
#we use data from the website 
#https://www.data.gov.au/data/dataset/geocoded-national-address-file-g-naf/

# --- Load only what's needed from G-NAF ---
addr = pd.read_csv(GNAF + "NSW_ADDRESS_DETAIL_psv.psv", sep="|", dtype=str,
                   usecols=["ADDRESS_DETAIL_PID", "LOCALITY_PID", "POSTCODE", "DATE_RETIRED"])
addr = addr[addr["DATE_RETIRED"].isna()].drop(columns="DATE_RETIRED")

geo = pd.read_csv(GNAF + "NSW_ADDRESS_DEFAULT_GEOCODE_psv.psv", sep="|", dtype=str,
                  usecols=["ADDRESS_DETAIL_PID", "LATITUDE", "LONGITUDE", "DATE_RETIRED"])
geo = geo[geo["DATE_RETIRED"].isna()].drop(columns="DATE_RETIRED")

loc = pd.read_csv(GNAF + "NSW_LOCALITY_psv.psv", sep="|", dtype=str,
                  usecols=["LOCALITY_PID", "LOCALITY_NAME"])

gnaf = addr.merge(geo, on="ADDRESS_DETAIL_PID").merge(loc, on="LOCALITY_PID")
gnaf["LATITUDE"] = gnaf["LATITUDE"].astype(float)
gnaf["LONGITUDE"] = gnaf["LONGITUDE"].astype(float)

def clean(s):
    return (s.fillna("").astype(str).str.upper().str.strip()
             .str.replace(r"\.0$", "", regex=True)
             .str.replace(r"[^A-Z0-9 ]", "", regex=True)
             .str.replace(r"\s+", " ", regex=True))

gnaf["k_suburb"] = clean(gnaf["LOCALITY_NAME"])
gnaf["k_post"] = clean(gnaf["POSTCODE"])

places = (gnaf.groupby(["k_suburb", "k_post"], as_index=False)[["LATITUDE", "LONGITUDE"]]
              .median()
              .rename(columns={"LATITUDE": "latitude", "LONGITUDE": "longitude"}))

# --- Your data ---
df = pd.read_csv("nsw_property_sales_sql_output.csv", dtype=str)
df = df.drop(columns=["latitude", "longitude"], errors="ignore")

df["k_suburb"] = clean(df["property_locality"])
df["k_post"] = clean(df["property_post_code"])

df = df.merge(places, on=["k_suburb", "k_post"], how="left")
print(f"Matched: {df['latitude'].notna().mean():.1%}")

df = df.drop(columns=["k_suburb", "k_post"])
df.duplicated().sum()
#df.to_csv("nsw_properties_geocoded.csv", index=False)

Matched: 100.0%


np.int64(0)

In [ ]:
pip install geopy